# Workload Scaling Analysis

This notebook analyzes how cost metrics vary across different workload levels to optimize dynamic hardware selection.

**Workload Levels Analyzed:**
- 10 requests/hour (Low utilization)
- 100 requests/hour (Medium utilization)
- 1000 requests/hour (High utilization)
- 5000 requests/hour (Peak utilization)

**Key Metrics:**
- Cost per request
- Hardware utilization %
- Cost crossover points
- Optimal hardware selection thresholds

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

In [ ]:
# Cost assumptions with idle multipliers
cost_assumptions = {
    ('cpu', 1): {'hourly_cost': 0.05, 'idle_multiplier': 1.0},
    ('cpu', 2): {'hourly_cost': 0.10, 'idle_multiplier': 1.0},
    ('cpu', 4): {'hourly_cost': 0.20, 'idle_multiplier': 1.0},
    ('cpu', 8): {'hourly_cost': 0.40, 'idle_multiplier': 1.0},
    ('cuda', 25): {'hourly_cost': 0.50, 'idle_multiplier': 0.8},
    ('cuda', 50): {'hourly_cost': 1.00, 'idle_multiplier': 0.8},
    ('cuda', 75): {'hourly_cost': 1.50, 'idle_multiplier': 0.8},
    ('cuda', 100): {'hourly_cost': 2.00, 'idle_multiplier': 0.8},
}

def get_device_key(record):
    variant = record['variant']
    if variant == 'cpu':
        return ('cpu', record['cpu_cores'])
    elif variant == 'cuda':
        return ('cuda', record['gpu_percentage'])

# Load and process Q4_K_M model data
with open('parsed_logs/night_logs_6_with_model_info.json', 'r') as f:
    data = json.load(f)

q4_models = [record for record in data if record.get('model_quant', '') == 'Q4_K_M']
print(f"Found {len(q4_models)} Q4_K_M model records")

# Process data
processed_data = []
for record in q4_models:
    try:
        device_key = get_device_key(record)
        if device_key not in cost_assumptions:
            continue

        costs = cost_assumptions[device_key]
        throughput = record.get('throughput_mean', 0)
        variant, config_value = device_key

        hw_config = f"CPU {config_value} cores" if variant == 'cpu' else f"GPU {config_value}%"

        processed_data.append({
            'hardware_config': hw_config,
            'variant': variant,
            'batch_size': record.get('concurrent_requests', 1),
            'throughput': throughput,
            'hourly_cost': costs['hourly_cost'],
            'idle_multiplier': costs['idle_multiplier'],
        })
    except Exception:
        continue

df = pd.DataFrame(processed_data)
print(f"Processed {len(df)} records")
print(f"Hardware configs: {sorted(df['hardware_config'].unique())}")

In [ ]:
# Calculate metrics across different workload levels
def calculate_workload_metrics(df, requests_per_hour, avg_tokens_per_request=100):
    results = []

    for hw_config in df['hardware_config'].unique():
        config_data = df[df['hardware_config'] == hw_config]
        avg_throughput = config_data['throughput'].mean()
        hourly_cost = config_data['hourly_cost'].iloc[0]
        idle_multiplier = config_data['idle_multiplier'].iloc[0]
        variant = config_data['variant'].iloc[0]

        # Calculate processing time
        time_per_request = avg_tokens_per_request / avg_throughput
        time_per_request_hours = time_per_request / 3600

        # Calculate utilization and costs
        total_processing_time_hours = requests_per_hour * time_per_request_hours
        idle_time_hours = max(0, 1.0 - total_processing_time_hours)

        processing_cost_per_hour = total_processing_time_hours * hourly_cost
        idle_cost_per_hour = idle_time_hours * hourly_cost * idle_multiplier
        total_hourly_cost = processing_cost_per_hour + idle_cost_per_hour

        cost_per_request = total_hourly_cost / requests_per_hour if requests_per_hour > 0 else 0
        utilization_percent = min(100, (total_processing_time_hours / 1.0) * 100)

        results.append({
            'requests_per_hour': requests_per_hour,
            'hardware_config': hw_config,
            'variant': variant,
            'cost_per_request': cost_per_request,
            'total_hourly_cost': total_hourly_cost,
            'utilization_percent': utilization_percent,
            'processing_cost_per_hour': processing_cost_per_hour,
            'idle_cost_per_hour': idle_cost_per_hour,
            'avg_throughput': avg_throughput,
        })

    return pd.DataFrame(results)

# Analyze multiple workload levels
workload_levels = [10, 50, 100, 200, 500, 1000, 2000, 5000]
all_workload_data = []

for rph in workload_levels:
    workload_result = calculate_workload_metrics(df, rph)
    all_workload_data.append(workload_result)

# Combine all workload data
combined_df = pd.concat(all_workload_data, ignore_index=True)

print(f"\nAnalyzed {len(workload_levels)} workload levels")
print(f"Total data points: {len(combined_df)}")
print(f"Workload range: {min(workload_levels)} - {max(workload_levels)} requests/hour")

In [ ]:
# Create color palette
configs = sorted(df['hardware_config'].unique())
cpu_configs = [c for c in configs if c.startswith('CPU')]
gpu_configs = [c for c in configs if c.startswith('GPU')]

cpu_colors = plt.cm.Oranges(np.linspace(0.4, 0.9, len(cpu_configs)))
gpu_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(gpu_configs)))

color_map = {}
for i, config in enumerate(cpu_configs):
    color_map[config] = cpu_colors[i]
for i, config in enumerate(gpu_configs):
    color_map[config] = gpu_colors[i]

# Visualization 1: Cost per Request vs Workload Level
plt.figure(figsize=(16, 10))

for hw_config in sorted(configs):
    config_data = combined_df[combined_df['hardware_config'] == hw_config]
    plt.plot(config_data['requests_per_hour'], config_data['cost_per_request'],
            marker='o', linewidth=3, markersize=8,
            label=hw_config, color=color_map[hw_config])

plt.xlabel('Requests per Hour', fontsize=14)
plt.ylabel('Cost per Request ($)', fontsize=14)
plt.title('Cost per Request vs Workload Level\nQ4_K_M Model - Dynamic Hardware Selection Optimization',
          fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.yscale('log')

# Add workload level annotations
for level in [10, 100, 1000]:
    plt.axvline(x=level, color='red', linestyle='--', alpha=0.5)
    plt.text(level, plt.ylim()[1]*0.8, f'{level}\nreq/hr', ha='center', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

print("\n📊 Key Insight: Cost crossover points clearly visible!")
print("CPU dominates at low workloads, GPU becomes efficient at high workloads.")

In [ ]:
# Visualization 2: Hardware Utilization vs Workload
plt.figure(figsize=(16, 8))

for hw_config in sorted(configs):
    config_data = combined_df[combined_df['hardware_config'] == hw_config]
    plt.plot(config_data['requests_per_hour'], config_data['utilization_percent'],
            marker='s', linewidth=3, markersize=8,
            label=hw_config, color=color_map[hw_config])

plt.xlabel('Requests per Hour', fontsize=14)
plt.ylabel('Hardware Utilization (%)', fontsize=14)
plt.title('Hardware Utilization vs Workload Level', fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xscale('log')

# Add utilization efficiency zones
plt.axhspan(0, 10, alpha=0.1, color='red', label='Under-utilized (<10%)')
plt.axhspan(10, 50, alpha=0.1, color='yellow', label='Moderate utilization (10-50%)')
plt.axhspan(50, 100, alpha=0.1, color='green', label='High utilization (>50%)')

plt.tight_layout()
plt.show()

print("\n⚡ Utilization Insight: GPUs need higher workloads to achieve efficient utilization.")

In [ ]:
# Find optimal hardware for each workload level
print("\n" + "="*80)
print("OPTIMAL HARDWARE SELECTION BY WORKLOAD LEVEL")
print("="*80)

optimal_selections = []
for rph in workload_levels:
    workload_data = combined_df[combined_df['requests_per_hour'] == rph]
    best_config = workload_data.loc[workload_data['cost_per_request'].idxmin()]

    optimal_selections.append({
        'requests_per_hour': rph,
        'optimal_hardware': best_config['hardware_config'],
        'cost_per_request': best_config['cost_per_request'],
        'utilization': best_config['utilization_percent'],
        'variant': best_config['variant']
    })

    print(f"{rph:4} req/hr: {best_config['hardware_config']:15} - "
          f"${best_config['cost_per_request']:.4f}/req ({best_config['utilization_percent']:5.1f}% util)")

optimal_df = pd.DataFrame(optimal_selections)

# Find CPU-GPU crossover point
cpu_workloads = optimal_df[optimal_df['variant'] == 'cpu']['requests_per_hour'].tolist()
gpu_workloads = optimal_df[optimal_df['variant'] == 'cuda']['requests_per_hour'].tolist()

if cpu_workloads and gpu_workloads:
    crossover_point = min(gpu_workloads)
    print(f"\n🎯 CPU-GPU CROSSOVER POINT: ~{crossover_point} requests/hour")
    print(f"   Below {crossover_point} req/hr: Use CPU")
    print(f"   Above {crossover_point} req/hr: Use GPU")

In [ ]:
# Visualization 3: Cost Heatmap
plt.figure(figsize=(14, 10))

# Create pivot table for heatmap
heatmap_data = combined_df.pivot_table(
    values='cost_per_request',
    index='hardware_config',
    columns='requests_per_hour',
    aggfunc='mean'
)

# Create heatmap
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='RdYlGn_r',
            cbar_kws={'label': 'Cost per Request ($)'})

plt.title('Cost per Request Heatmap\nHardware Config vs Workload Level',
          fontsize=16, fontweight='bold')
plt.xlabel('Requests per Hour', fontsize=14)
plt.ylabel('Hardware Configuration', fontsize=14)
plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

print("\n🔥 Heatmap shows clear cost patterns across workload levels!")
print("Green = Lower cost, Red = Higher cost")

In [ ]:
# Calculate cost savings from optimal selection
print("\n" + "="*70)
print("COST SAVINGS FROM DYNAMIC HARDWARE SELECTION")
print("="*70)

# Compare optimal selection vs always using best CPU or best GPU
always_cpu_config = 'CPU 1 cores'  # Cheapest CPU
always_gpu_config = 'GPU 25%'     # Cheapest GPU

savings_analysis = []
for rph in workload_levels:
    workload_data = combined_df[combined_df['requests_per_hour'] == rph]

    optimal_cost = workload_data['cost_per_request'].min()
    cpu_cost = workload_data[workload_data['hardware_config'] == always_cpu_config]['cost_per_request'].iloc[0]
    gpu_cost = workload_data[workload_data['hardware_config'] == always_gpu_config]['cost_per_request'].iloc[0]

    cpu_savings = ((cpu_cost - optimal_cost) / cpu_cost) * 100 if cpu_cost > optimal_cost else 0
    gpu_savings = ((gpu_cost - optimal_cost) / gpu_cost) * 100 if gpu_cost > optimal_cost else 0

    savings_analysis.append({
        'requests_per_hour': rph,
        'optimal_cost': optimal_cost,
        'cpu_cost': cpu_cost,
        'gpu_cost': gpu_cost,
        'savings_vs_cpu': cpu_savings,
        'savings_vs_gpu': gpu_savings
    })

    print(f"{rph:4} req/hr: Optimal=${optimal_cost:.4f} | "
          f"vs CPU: {cpu_savings:5.1f}% savings | "
          f"vs GPU: {gpu_savings:5.1f}% savings")

savings_df = pd.DataFrame(savings_analysis)

# Calculate average savings
avg_cpu_savings = savings_df['savings_vs_cpu'].mean()
avg_gpu_savings = savings_df['savings_vs_gpu'].mean()

print(f"\n💰 AVERAGE COST SAVINGS:")
print(f"   vs Always CPU: {avg_cpu_savings:.1f}%")
print(f"   vs Always GPU: {avg_gpu_savings:.1f}%")
print(f"\n🎯 Dynamic selection provides significant cost optimization!")

In [ ]:
# Summary and Recommendations
print("\n" + "="*80)
print("DYNAMIC HARDWARE SELECTION RECOMMENDATIONS")
print("="*80)

print(f"\n📊 KEY METRICS TO MONITOR:")
print(f"1. Current requests/hour (15-min rolling average)")
print(f"2. Cost per request")
print(f"3. Hardware utilization percentage")
print(f"4. Queue depth and response time")

print(f"\n🎯 OPTIMIZATION THRESHOLDS:")
if cpu_workloads and gpu_workloads:
    print(f"• Low workload (<{crossover_point} req/hr): Use CPU configurations")
    print(f"• High workload (>{crossover_point} req/hr): Use GPU configurations")
    print(f"• Transition zone ({crossover_point//2}-{crossover_point*2} req/hr): Monitor and switch dynamically")

print(f"\n⚡ IMPLEMENTATION STRATEGY:")
print(f"1. Monitor workload patterns in real-time")
print(f"2. Predict request volume using rolling averages")
print(f"3. Switch hardware when cost savings > 10%")
print(f"4. Add hysteresis to prevent frequent switching")
print(f"5. Consider switching costs in decision algorithm")

print(f"\n💡 EXPECTED BENEFITS:")
print(f"• Average {avg_cpu_savings:.0f}% cost savings vs static CPU")
print(f"• Average {avg_gpu_savings:.0f}% cost savings vs static GPU")
print(f"• Optimal resource utilization across workload levels")
print(f"• Automatic adaptation to changing demand patterns")

# Save results
combined_df.to_csv('workload_scaling_analysis_results.csv', index=False)
optimal_df.to_csv('optimal_hardware_selections.csv', index=False)
print(f"\n💾 Results saved to CSV files for system implementation")